# CS781 Phase 1 — Mamba legal language-model training

A compact training notebook reusing the existing Mamba architecture and legal-corpus streaming pipeline from the original notebook, with resumable checkpoints and optional Kaggle Dataset version uploads.

**Before running:** enable a GPU and attach these two Kaggle datasets as inputs:
- `oipranesh/cs781-phase4-training-prep`
- `oipranesh/cs781-phase4-final-recovery`

The notebook does not load or modify the GPT-2 checkpoint. Mamba starts from random initialization. The first run installs/import-checks Mamba, validates the data pipeline, and trains. Checkpoints are saved locally every 25 optimizer steps and Kaggle Dataset uploads are attempted every 250 steps if credentials are configured.

**Important:** Kaggle cannot guarantee a final save when a session is forcibly terminated. Frequent local saves plus periodic uploads reduce the risk.


In [ ]:
# 1. Environment and imports
import os, sys, gc, json, math, time, shutil, signal, subprocess, random
from pathlib import Path
import torch
from torch.utils.data import DataLoader, IterableDataset, get_worker_info
from tokenizers import Tokenizer

assert torch.cuda.is_available(), "Enable a GPU accelerator in Kaggle before training."
DEVICE = torch.device("cuda:0")
print("Python:", sys.version)
print("PyTorch:", torch.__version__, "| CUDA:", torch.version.cuda)
print("GPU:", torch.cuda.get_device_name(0))


In [ ]:
# ============================================================
# MAMBA — CANONICAL STREAMING CONFIGURATION
# ============================================================

import os
import json
import gc
import torch

from pathlib import Path
from tokenizers import Tokenizer
from torch.utils.data import DataLoader, IterableDataset, get_worker_info


# ============================================================
# KAGGLE DATASET ROOTS
# ============================================================

TRAINING_PREP_ROOT = Path(
    "/kaggle/input/datasets/oipranesh/"
    "cs781-phase4-training-prep"
)

FINAL_RECOVERY_ROOT = Path(
    "/kaggle/input/datasets/oipranesh/"
    "cs781-phase4-final-recovery"
)


# ============================================================
# CANONICAL ARTIFACTS
# ============================================================

TRAIN_MANIFEST = (
    TRAINING_PREP_ROOT
    / "manifests"
    / "train_manifest.jsonl"
)

VAL_MANIFEST = (
    TRAINING_PREP_ROOT
    / "manifests"
    / "validation_manifest.jsonl"
)

TEST_MANIFEST = (
    TRAINING_PREP_ROOT
    / "manifests"
    / "test_manifest.jsonl"
)

TOKENIZER_PATH = (
    TRAINING_PREP_ROOT
    / "tokenizer"
    / "legal_bpe_32k.json"
)

INDEX_PATH = (
    FINAL_RECOVERY_ROOT
    / "indexes"
    / "source_index.jsonl"
)

CONTEXT_LENGTH = 4096
BATCH_SIZE = 1
SEED = 42


# ============================================================
# VALIDATE
# ============================================================

print("=" * 70)
print("MAMBA — CANONICAL STREAMING CONFIGURATION")
print("=" * 70)

paths = {
    "Training prep": TRAINING_PREP_ROOT,
    "Final recovery": FINAL_RECOVERY_ROOT,
    "Train manifest": TRAIN_MANIFEST,
    "Validation manifest": VAL_MANIFEST,
    "Test manifest": TEST_MANIFEST,
    "Tokenizer": TOKENIZER_PATH,
    "Source index": INDEX_PATH,
}

for name, path in paths.items():

    print(
        f"{name:<22}: "
        f"{'✓' if path.exists() else '✗'} "
        f"{path}"
    )

    assert path.exists(), f"Missing: {path}"


print()
print("Context length:", CONTEXT_LENGTH)
print("Batch size    :", BATCH_SIZE)
print("Seed          :", SEED)

print()
print("=" * 70)
print("✓ CANONICAL STREAMING ARTIFACTS VERIFIED")
print("=" * 70)

In [ ]:
# ============================================================
# MAMBA — LOAD CANONICAL TOKENIZER
# ============================================================

tokenizer = Tokenizer.from_file(
    str(TOKENIZER_PATH)
)

VOCAB_SIZE = tokenizer.get_vocab_size()

print("=" * 70)
print("MAMBA — TOKENIZER")
print("=" * 70)

print("Path       :", TOKENIZER_PATH)
print("Vocabulary :", VOCAB_SIZE)

assert VOCAB_SIZE == 32000

for token in [
    "[PAD]",
    "[UNK]",
    "[BOS]",
    "[EOS]",
]:

    token_id = tokenizer.token_to_id(token)

    print(
        f"{token:<8}: {token_id}"
    )

    assert token_id is not None

print()
print("✓ Canonical 32K tokenizer loaded")

In [ ]:
%pip install -q --no-build-isolation --no-deps \
    "mamba-ssm==2.3.2.post1"

In [ ]:
import os
import subprocess
import torch
import sys

print("Python:", sys.version)
print("PyTorch:", torch.__version__)
print("CUDA:", torch.version.cuda)

print("\nGPU status:")
print(subprocess.run(
    ["nvidia-smi"], capture_output=True, text=True
).stdout[:2500])

print("\nMamba package status:")
result = subprocess.run(
    [sys.executable, "-m", "pip", "show", "mamba-ssm"],
    capture_output=True, text=True
)
print(result.stdout or result.stderr)

In [ ]:
import torch
from mamba_ssm import Mamba

model = Mamba(
    d_model=768,
    d_state=16,
    d_conv=4,
    expand=2,
).cuda()

x = torch.randn(1, 128, 768, device="cuda")
y = model(x)

print("Output shape:", y.shape)
assert y.shape == x.shape
print("Mamba smoke test passed.")

In [ ]:
import torch
import sys

print("Mamba modules already loaded:")
print([name for name in sys.modules if "mamba" in name.lower()][:30])

print("\nRelevant installed packages:")
for name in ["mamba-ssm", "causal-conv1d", "triton"]:
    result = __import__("subprocess").run(
        [sys.executable, "-m", "pip", "show", name],
        capture_output=True, text=True
    )
    print(f"\n{name}:")
    print(result.stdout[:700] or "Not installed")

In [ ]:
# ============================================================
# MAMBA EXPERIMENT — LANGUAGE MODEL ARCHITECTURE
# NO DATASET / CHECKPOINT MODIFICATION
# ============================================================

import gc
import torch
import torch.nn as nn

from mamba_ssm import Mamba

# ============================================================
# CANONICAL EXPERIMENT CONFIG
# ============================================================

DEVICE = torch.device("cuda:0")

VOCAB_SIZE = 32000
CONTEXT_LENGTH = 4096

# Match the existing Phase-4 Transformer width/depth.
N_LAYER = 12
D_MODEL = 768

# Mamba-specific configuration.
D_STATE = 16
D_CONV = 4
EXPAND = 2

# ============================================================
# MAMBA LANGUAGE MODEL
# ============================================================

class MambaBlock(nn.Module):

    def __init__(
        self,
        d_model,
        d_state,
        d_conv,
        expand,
    ):
        super().__init__()

        self.norm = nn.RMSNorm(d_model)

        self.mixer = Mamba(
            d_model=d_model,
            d_state=d_state,
            d_conv=d_conv,
            expand=expand,
        )

    def forward(self, x):
        return x + self.mixer(self.norm(x))


class MambaLanguageModel(nn.Module):

    def __init__(
        self,
        vocab_size,
        d_model,
        n_layer,
        d_state,
        d_conv,
        expand,
    ):
        super().__init__()

        self.vocab_size = vocab_size
        self.d_model = d_model
        self.n_layer = n_layer

        self.token_embedding = nn.Embedding(
            vocab_size,
            d_model,
        )

        self.layers = nn.ModuleList([
            MambaBlock(
                d_model=d_model,
                d_state=d_state,
                d_conv=d_conv,
                expand=expand,
            )
            for _ in range(n_layer)
        ])

        self.final_norm = nn.RMSNorm(d_model)

        self.lm_head = nn.Linear(
            d_model,
            vocab_size,
            bias=False,
        )

        # Weight tying — same token embedding and LM head weights.
        self.lm_head.weight = self.token_embedding.weight

    def forward(
        self,
        input_ids,
        labels=None,
    ):

        x = self.token_embedding(input_ids)

        for layer in self.layers:
            x = layer(x)

        x = self.final_norm(x)

        logits = self.lm_head(x)

        loss = None

        if labels is not None:

            # Causal next-token prediction: position t predicts token t+1.
            shift_logits = logits[:, :-1, :].contiguous()
            shift_labels = labels[:, 1:].contiguous()
            loss = nn.functional.cross_entropy(
                shift_logits.reshape(-1, self.vocab_size),
                shift_labels.reshape(-1),
                ignore_index=-100,
            )

        return {
            "logits": logits,
            "loss": loss,
        }


# ============================================================
# CONSTRUCT
# ============================================================

mamba_model = MambaLanguageModel(
    vocab_size=VOCAB_SIZE,
    d_model=D_MODEL,
    n_layer=N_LAYER,
    d_state=D_STATE,
    d_conv=D_CONV,
    expand=EXPAND,
).to(DEVICE)

print("=" * 70)
print("MAMBA LANGUAGE MODEL CREATED")
print("=" * 70)

print("Device          :", DEVICE)
print("Vocabulary      :", VOCAB_SIZE)
print("Context         :", CONTEXT_LENGTH)
print("Layers          :", N_LAYER)
print("Model dimension :", D_MODEL)
print("d_state         :", D_STATE)
print("d_conv          :", D_CONV)
print("expand          :", EXPAND)

In [ ]:
# ============================================================
# MAMBA — DOCUMENT-AWARE STREAMING PACKER
# SAME IMPLEMENTATION AS PHASE 4
# ============================================================

import os
import json
import torch

from torch.utils.data import IterableDataset


class DocumentAwarePacker(IterableDataset):

    def __init__(
        self,
        manifest_path,
        index_path,
        tokenizer,
        context_length=4096,
        checkpoint=str(TRAINING_PREP_ROOT),
        split="train",
    ):
        super().__init__()

        self.manifest_path = str(manifest_path)
        self.index_path = str(index_path)
        self.tokenizer = tokenizer
        self.context_length = int(context_length)
        self.checkpoint = str(checkpoint)
        self.split = split

        # ----------------------------------------------------
        # LOAD BYTE-OFFSET INDEX
        # ----------------------------------------------------

        self.index = {}

        with open(
            self.index_path,
            "r",
            encoding="utf-8",
        ) as f:

            for line in f:

                line = line.strip()

                if not line:
                    continue

                rec = json.loads(line)

                self.index[
                    rec["text_hash"]
                ] = {
                    "source_file":
                        rec["source_file"],

                    "byte_offset":
                        rec["byte_offset"],

                    "byte_length":
                        rec["byte_length"],
                }

        print(
            f"Loaded index entries: "
            f"{len(self.index):,}"
        )

        # ----------------------------------------------------
        # LOAD MANIFEST
        # ----------------------------------------------------

        self.manifest = []

        with open(
            self.manifest_path,
            "r",
            encoding="utf-8",
        ) as f:

            for line in f:

                line = line.strip()

                if not line:
                    continue

                rec = json.loads(line)

                self.manifest.append(rec)

        print(
            f"Loaded manifest records: "
            f"{len(self.manifest):,}"
        )

        # ----------------------------------------------------
        # SPECIAL TOKENS
        # ----------------------------------------------------

        self.bos_id = self._get_token_id("[BOS]")
        self.eos_id = self._get_token_id("[EOS]")
        self.pad_id = self._get_token_id("[PAD]")

        print("BOS id :", self.bos_id)
        print("EOS id :", self.eos_id)
        print("PAD id :", self.pad_id)

    # --------------------------------------------------------
    # TOKENIZER COMPATIBILITY
    # --------------------------------------------------------

    def _get_token_id(self, token):

        token_id = None

        if hasattr(
            self.tokenizer,
            "token_to_id",
        ):

            token_id = self.tokenizer.token_to_id(
                token
            )

        elif hasattr(
            self.tokenizer,
            "convert_tokens_to_ids",
        ):

            token_id = (
                self.tokenizer
                .convert_tokens_to_ids(token)
            )

        if token_id is None:

            raise ValueError(
                "Tokenizer does not contain "
                f"required special token: {token}"
            )

        if isinstance(token_id, list):

            if len(token_id) != 1:

                raise ValueError(
                    f"Unexpected token id for "
                    f"{token}: {token_id}"
                )

            token_id = token_id[0]

        return int(token_id)

    # --------------------------------------------------------
    # RESOLVE SOURCE DOCUMENT
    # --------------------------------------------------------

    def _read_source_record(self, rec):

        text_hash = rec["text_hash"]

        if text_hash not in self.index:

            raise KeyError(
                "text_hash not found in "
                f"source index: {text_hash}"
            )

        idx = self.index[text_hash]

        relative_source = idx["source_file"]

        source_name = relative_source

        if source_name.endswith(".gz"):

            source_name = source_name[:-3]

        source_path = os.path.join(
            self.checkpoint,
            "dedup",
            source_name,
        )

        if not os.path.exists(source_path):

            raise FileNotFoundError(
                "Resolved source does not exist:\n"
                f"{source_path}"
            )

        with open(
            source_path,
            "rb",
        ) as f:

            f.seek(idx["byte_offset"])

            raw = f.read(
                idx["byte_length"]
            )

        raw = raw.rstrip(
            b"\r\n"
        )

        source_record = json.loads(
            raw.decode("utf-8")
        )

        return source_record

    # --------------------------------------------------------
    # EXTRACT TEXT
    # --------------------------------------------------------

    def _extract_text(self, source_record):

        if isinstance(
            source_record,
            str,
        ):

            return source_record

        for key in (
            "text",
            "content",
            "document",
            "body",
        ):

            value = source_record.get(key)

            if isinstance(
                value,
                str,
            ):

                return value

        raise KeyError(
            "Could not find document text field. "
            f"Available fields: "
            f"{list(source_record.keys())}"
        )

    # --------------------------------------------------------
    # TOKENIZE DOCUMENT
    # --------------------------------------------------------

    def _tokenize_document(self, text):

        if hasattr(
            self.tokenizer,
            "encode",
        ):

            encoded = self.tokenizer.encode(
                text
            )

            if hasattr(
                encoded,
                "ids",
            ):

                token_ids = encoded.ids

            elif isinstance(
                encoded,
                dict,
            ):

                token_ids = encoded[
                    "input_ids"
                ]

            else:

                token_ids = encoded

        else:

            raise TypeError(
                "Unsupported tokenizer interface."
            )

        token_ids = list(
            map(int, token_ids)
        )

        # Explicit document boundaries.
        token_ids = (
            [self.bos_id]
            + token_ids
            + [self.eos_id]
        )

        return token_ids

    # --------------------------------------------------------
    # DOCUMENT-AWARE PACKING
    # --------------------------------------------------------

    def _pack_document(self, token_ids):

        if len(token_ids) > self.context_length:

            for start in range(
                0,
                len(token_ids),
                self.context_length,
            ):

                chunk = token_ids[
                    start:
                    start + self.context_length
                ]

                if len(chunk) < self.context_length:

                    chunk = (
                        chunk
                        + [self.pad_id]
                        * (
                            self.context_length
                            - len(chunk)
                        )
                    )

                yield chunk

            return

        chunk = token_ids[:]

        if len(chunk) < self.context_length:

            chunk = (
                chunk
                + [self.pad_id]
                * (
                    self.context_length
                    - len(chunk)
                )
            )

        yield chunk

    # --------------------------------------------------------
    # STREAM
    # --------------------------------------------------------

    def __iter__(self):

        for rec in self.manifest:

            token_ids = self._tokenize_document(
                self._extract_text(
                    self._read_source_record(rec)
                )
            )

            for packed in self._pack_document(
                token_ids
            ):

                input_ids = torch.tensor(
                    packed,
                    dtype=torch.long,
                )

                labels = input_ids.clone()

                labels[
                    labels == self.pad_id
                ] = -100

                attention_mask = (
                    input_ids != self.pad_id
                ).long()

                yield {
                    "input_ids":
                        input_ids,

                    "labels":
                        labels,

                    "attention_mask":
                        attention_mask,
                }


print("=" * 70)
print("✓ DOCUMENT-AWARE PACKER DEFINED")
print("=" * 70)

In [ ]:
# ============================================================
# MAMBA — WORKER-SAFE STREAMING DATASET
# SAME PHASE-4 IMPLEMENTATION
# ============================================================

class WorkerSafeDocumentAwarePacker(
    DocumentAwarePacker
):

    def __iter__(self):

        worker_info = get_worker_info()

        # ----------------------------------------------------
        # SINGLE PROCESS
        # ----------------------------------------------------

        if worker_info is None:

            start = 0
            step = 1

        # ----------------------------------------------------
        # MULTI-WORKER
        # ----------------------------------------------------

        else:

            worker_id = worker_info.id
            num_workers = worker_info.num_workers

            start = worker_id
            step = num_workers

        # ----------------------------------------------------
        # STRIDED MANIFEST PARTITION
        # ----------------------------------------------------

        for manifest_idx in range(
            start,
            len(self.manifest),
            step,
        ):

            rec = self.manifest[
                manifest_idx
            ]

            source_record = (
                self._read_source_record(rec)
            )

            text = self._extract_text(
                source_record
            )

            token_ids = (
                self._tokenize_document(text)
            )

            for packed in self._pack_document(
                token_ids
            ):

                input_ids = torch.tensor(
                    packed,
                    dtype=torch.long,
                )

                labels = input_ids.clone()

                labels[
                    labels == self.pad_id
                ] = -100

                attention_mask = (
                    input_ids != self.pad_id
                ).long()

                yield {
                    "input_ids":
                        input_ids,

                    "labels":
                        labels,

                    "attention_mask":
                        attention_mask,
                }


print("=" * 70)
print("✓ WORKER-SAFE PACKER DEFINED")
print("=" * 70)

In [ ]:
# ============================================================
# MAMBA — CREATE STREAMING TRAIN DATASET
# NO TOKENIZED CORPUS CREATED
# ============================================================

gc.collect()

train_dataset = WorkerSafeDocumentAwarePacker(
    manifest_path=TRAIN_MANIFEST,
    index_path=INDEX_PATH,
    tokenizer=tokenizer,
    context_length=CONTEXT_LENGTH,
    checkpoint=str(TRAINING_PREP_ROOT),
    split="train",
)

print("=" * 70)
print("MAMBA — TRAINING DATASET")
print("=" * 70)

print("Dataset type :", type(train_dataset).__name__)
print("Manifest     :", TRAIN_MANIFEST)
print("Index        :", INDEX_PATH)
print("Tokenizer    :", TOKENIZER_PATH)
print("Context      :", CONTEXT_LENGTH)
print("Split        :", "train")

print()
print("Manifest records:")
print(f"{len(train_dataset.manifest):,}")

print()
print("Index records:")
print(f"{len(train_dataset.index):,}")

assert len(train_dataset.manifest) == 3_020_655

print()
print("✓ Streaming dataset created")
print("✓ No tokenized corpus materialized")

In [ ]:
# ============================================================
# MAMBA — SINGLE-WORKER STREAMING BATCH TEST
# READ-ONLY
# ============================================================

import time

train_loader = DataLoader(
    train_dataset,
    batch_size=1,
    num_workers=0,
    pin_memory=True,
)

print("=" * 70)
print("MAMBA — FIRST REAL TRAINING BATCH")
print("=" * 70)

t0 = time.time()

batch = next(iter(train_loader))

elapsed = time.time() - t0

print("Load time:", round(elapsed, 3), "seconds")

print()
print("BATCH")
print("-" * 70)

for key, value in batch.items():

    print(
        f"{key:<18}: "
        f"shape={tuple(value.shape)} "
        f"dtype={value.dtype}"
    )

# ------------------------------------------------------------
# SHAPE CHECKS
# ------------------------------------------------------------

assert batch["input_ids"].shape == (
    1,
    CONTEXT_LENGTH,
)

assert batch["labels"].shape == (
    1,
    CONTEXT_LENGTH,
)

assert batch["attention_mask"].shape == (
    1,
    CONTEXT_LENGTH,
)

# ------------------------------------------------------------
# VALUE CHECKS
# ------------------------------------------------------------

assert batch["input_ids"].dtype == torch.long
assert batch["labels"].dtype == torch.long
assert batch["attention_mask"].dtype == torch.long

assert torch.all(
    (batch["input_ids"] >= 0)
    & (batch["input_ids"] < VOCAB_SIZE)
)

assert torch.all(
    (batch["attention_mask"] == 0)
    | (batch["attention_mask"] == 1)
)

print()
print("✓ Shapes correct")
print("✓ Dtypes correct")
print("✓ Token IDs within vocabulary")
print("✓ Attention mask valid")

print()
print("=" * 70)
print("✓ REAL STREAMING BATCH PASSED")
print("=" * 70)

## Training and checkpointing

The defaults are deliberately conservative for a first Mamba run. Increase `TOTAL_STEPS` after confirming that loss decreases and checkpoint uploads work. `KAGGLE_DATASET_SLUG` must be a valid, unique slug you own; the notebook tries to create the private dataset if versioning fails because it does not yet exist.

If you prefer not to upload checkpoints, leave Kaggle API secrets unconfigured: local checkpoints will still be saved under `/kaggle/working/mamba_phase1/checkpoints`.


In [ ]:
# 10. Training configuration, optimizer, and Kaggle credentials
TOTAL_STEPS = 3000
LEARNING_RATE = 5e-5
WEIGHT_DECAY = 0.01
GRAD_ACCUM = 8
MAX_GRAD_NORM = 1.0
LOG_EVERY = 10
LOCAL_CHECKPOINT_EVERY = 25
KAGGLE_UPLOAD_EVERY = 250
BATCH_SIZE = 1
NUM_WORKERS = 0  # robust setting for this streaming dataset

OUTPUT_DIR = Path("/kaggle/working/mamba_phase1")
CHECKPOINT_DIR = OUTPUT_DIR / "checkpoints"
CHECKPOINT_DIR.mkdir(parents=True, exist_ok=True)
LOG_PATH = OUTPUT_DIR / "training_log.jsonl"
STATE_PATH = OUTPUT_DIR / "training_state.json"
LATEST_PATH = CHECKPOINT_DIR / "checkpoint_latest.pt"

KAGGLE_DATASET_SLUG = "oipranesh/cs781-phase1-mamba-checkpoints"
UPLOAD_DIR = Path("/kaggle/working/mamba_checkpoint_dataset")
UPLOAD_DIR.mkdir(parents=True, exist_ok=True)

try:
    from kaggle_secrets import UserSecretsClient
    secrets = UserSecretsClient()
    os.environ["KAGGLE_USERNAME"] = secrets.get_secret("KAGGLE_USERNAME")
    os.environ["KAGGLE_KEY"] = secrets.get_secret("KAGGLE_KEY")
    print("Kaggle API credentials loaded.")
except Exception as e:
    print("Kaggle Dataset upload unavailable until KAGGLE_USERNAME and KAGGLE_KEY are configured in Kaggle Secrets.")
    print("Local checkpoint saving remains enabled.")

optimizer = torch.optim.AdamW(
    mamba_model.parameters(),
    lr=LEARNING_RATE,
    weight_decay=WEIGHT_DECAY,
)
USE_AMP = True
AMP_DTYPE = torch.bfloat16 if torch.cuda.is_bf16_supported() else torch.float16
scaler = torch.amp.GradScaler("cuda", enabled=(AMP_DTYPE == torch.float16))
print("Target steps:", TOTAL_STEPS, "| grad accumulation:", GRAD_ACCUM, "| AMP:", AMP_DTYPE)


In [ ]:
# 11. Checkpoint discovery, atomic save, and Kaggle Dataset upload
import re

MODEL_CONFIG = {
    "architecture": "MambaLanguageModel",
    "vocab_size": VOCAB_SIZE,
    "context_length": CONTEXT_LENGTH,
    "d_model": D_MODEL,
    "n_layer": N_LAYER,
    "d_state": D_STATE,
    "d_conv": D_CONV,
    "expand": EXPAND,
    "weight_tying": True,
}

def find_resume_checkpoint():
    candidates = []
    if LATEST_PATH.exists():
        candidates.append(LATEST_PATH)
    for p in Path("/kaggle/input").rglob("checkpoint_latest.pt"):
        candidates.append(p)
    for p in Path("/kaggle/input").rglob("mamba_checkpoint_step_*.pt"):
        candidates.append(p)
    ranked = []
    for p in dict.fromkeys(candidates):
        try:
            ck = torch.load(p, map_location="cpu", weights_only=False)
            if ck.get("model_config") != MODEL_CONFIG:
                print("Skipping incompatible checkpoint:", p)
                continue
            ranked.append((int(ck.get("step", -1)), p))
        except Exception as e:
            print("Skipping unreadable checkpoint:", p, repr(e))
    if not ranked:
        return None
    ranked.sort(key=lambda x: x[0])
    print(f"Resume candidate: step {ranked[-1][0]} at {ranked[-1][1]}")
    return ranked[-1][1]

def save_checkpoint(step, reason="periodic"):
    tmp = CHECKPOINT_DIR / "checkpoint_tmp.pt"
    payload = {
        "step": int(step),
        "model_state_dict": mamba_model.state_dict(),
        "optimizer_state_dict": optimizer.state_dict(),
        "scaler_state_dict": scaler.state_dict(),
        "model_config": MODEL_CONFIG,
        "reason": reason,
        "saved_unix_time": time.time(),
    }
    torch.save(payload, tmp)
    os.replace(tmp, LATEST_PATH)
    # A numbered snapshot allows recovery if the rolling file is interrupted.
    numbered = CHECKPOINT_DIR / f"mamba_checkpoint_step_{step:06d}.pt"
    shutil.copy2(LATEST_PATH, numbered)
    snapshots = sorted(CHECKPOINT_DIR.glob("mamba_checkpoint_step_*.pt"))
    for old in snapshots[:-2]:
        try: old.unlink()
        except OSError: pass
    STATE_PATH.write_text(json.dumps({
        "step": int(step), "reason": reason, "saved_unix_time": time.time(),
        "model_config": MODEL_CONFIG
    }, indent=2))
    print(f"Saved step {step}: {LATEST_PATH} ({LATEST_PATH.stat().st_size / 1e9:.2f} GB)")

def upload_checkpoint_to_kaggle(step):
    if not (os.environ.get("KAGGLE_USERNAME") and os.environ.get("KAGGLE_KEY")):
        print("Skipping upload: Kaggle API credentials are not configured.")
        return False
    meta = UPLOAD_DIR / "dataset-metadata.json"
    if not meta.exists():
        meta.write_text(json.dumps({
            "title": "CS781 Phase 1 Mamba Checkpoints",
            "id": KAGGLE_DATASET_SLUG,
            "licenses": [{"name": "CC0-1.0"}],
            "private": True
        }, indent=2))
    shutil.copy2(LATEST_PATH, UPLOAD_DIR / "checkpoint_latest.pt")
    shutil.copy2(STATE_PATH, UPLOAD_DIR / "training_state.json")
    if LOG_PATH.exists():
        shutil.copy2(LOG_PATH, UPLOAD_DIR / "training_log.jsonl")
    command = ["kaggle", "datasets", "version", "-p", str(UPLOAD_DIR),
               "-m", f"CS781 Mamba checkpoint step {step}", "--dir-mode", "tar"]
    try:
        result = subprocess.run(command, check=True, text=True, capture_output=True, timeout=1800)
        print(result.stdout[-2000:] if result.stdout else "Kaggle Dataset version uploaded.")
        return True
    except Exception as e:
        print("Dataset version failed; attempting initial private Dataset creation.", repr(e))
        create_cmd = ["kaggle", "datasets", "create", "-p", str(UPLOAD_DIR), "--dir-mode", "tar"]
        try:
            result = subprocess.run(create_cmd, check=True, text=True, capture_output=True, timeout=1800)
            print(result.stdout[-2000:] if result.stdout else "Kaggle Dataset created.")
            return True
        except Exception as e2:
            print("Upload failed; local checkpoint remains safe:", repr(e2))
            if getattr(e2, "stderr", None): print(e2.stderr[-2000:])
            return False

RESUME_PATH = find_resume_checkpoint()
START_STEP = 0
if RESUME_PATH is not None:
    ckpt = torch.load(RESUME_PATH, map_location="cpu", weights_only=False)
    mamba_model.load_state_dict(ckpt["model_state_dict"], strict=True)
    if "optimizer_state_dict" in ckpt:
        optimizer.load_state_dict(ckpt["optimizer_state_dict"])
    if "scaler_state_dict" in ckpt:
        scaler.load_state_dict(ckpt["scaler_state_dict"])
    START_STEP = int(ckpt["step"])
    print("Resumed from step", START_STEP)
else:
    print("No compatible Mamba checkpoint found; training from random initialization.")


In [ ]:
# 12. Train; save locally every 25 steps and upload every 250 steps
mamba_model.train()
train_iterator = iter(train_loader)
running_loss = 0.0
logged_steps = 0
run_started = time.time()
current_step = START_STEP

def emergency_save(signum, frame):
    print(f"Received signal {signum}; attempting checkpoint save.")
    try:
        save_checkpoint(current_step, reason=f"signal_{signum}")
    finally:
        raise SystemExit(128 + signum)

try:
    signal.signal(signal.SIGTERM, emergency_save)
    signal.signal(signal.SIGINT, emergency_save)
except Exception:
    pass

for step in range(START_STEP + 1, TOTAL_STEPS + 1):
    current_step = step
    optimizer.zero_grad(set_to_none=True)
    loss_sum = 0.0
    for micro in range(GRAD_ACCUM):
        try:
            batch = next(train_iterator)
        except StopIteration:
            train_iterator = iter(train_loader)
            batch = next(train_iterator)

        input_ids = batch["input_ids"].to(DEVICE, dtype=torch.long, non_blocking=True)
        labels = batch["labels"].to(DEVICE, dtype=torch.long, non_blocking=True)
        with torch.autocast(device_type="cuda", dtype=AMP_DTYPE, enabled=USE_AMP):
            outputs = mamba_model(input_ids=input_ids, labels=labels)
            loss = outputs["loss"]
            if not torch.isfinite(loss):
                raise RuntimeError(f"Non-finite loss at step {step}: {loss.item()}")
            scaled_loss = loss / GRAD_ACCUM
        if AMP_DTYPE == torch.float16:
            scaler.scale(scaled_loss).backward()
        else:
            scaled_loss.backward()
        loss_sum += float(loss.detach().float().item())

    if AMP_DTYPE == torch.float16:
        scaler.unscale_(optimizer)
    grad_norm = torch.nn.utils.clip_grad_norm_(mamba_model.parameters(), MAX_GRAD_NORM)
    if not torch.isfinite(torch.as_tensor(grad_norm)):
        raise RuntimeError(f"Non-finite gradient norm at step {step}")
    if AMP_DTYPE == torch.float16:
        scaler.step(optimizer)
        scaler.update()
    else:
        optimizer.step()
    mean_loss = loss_sum / GRAD_ACCUM
    running_loss += mean_loss
    logged_steps += 1

    if step % LOG_EVERY == 0 or step == START_STEP + 1:
        avg = running_loss / max(logged_steps, 1)
        rec = {"step": step, "loss": avg, "perplexity": math.exp(min(avg, 20)),
               "grad_norm": float(grad_norm), "elapsed_seconds": time.time()-run_started}
        with LOG_PATH.open("a", encoding="utf-8") as f:
            f.write(json.dumps(rec) + "\n")
        print(f"step={step}/{TOTAL_STEPS} loss={avg:.4f} ppl={rec['perplexity']:.2f} "
              f"grad={float(grad_norm):.3f} elapsed_min={(time.time()-run_started)/60:.1f}")
        running_loss, logged_steps = 0.0, 0

    if step % LOCAL_CHECKPOINT_EVERY == 0 or step == TOTAL_STEPS:
        save_checkpoint(step, reason="final" if step == TOTAL_STEPS else "periodic")
    if step % KAGGLE_UPLOAD_EVERY == 0 or step == TOTAL_STEPS:
        if step % LOCAL_CHECKPOINT_EVERY != 0 and step != TOTAL_STEPS:
            save_checkpoint(step, reason="upload_boundary")
        upload_checkpoint_to_kaggle(step)

print("Training finished at step", current_step)
print("Latest checkpoint:", LATEST_PATH)


## After a Kaggle session ends

1. Check the last successful `Saved step ...` and `Kaggle Dataset` upload messages.
2. Attach the private Mamba checkpoint dataset to the next notebook session.
3. Run from the top. The notebook searches attached inputs for a compatible `checkpoint_latest.pt` and restores model, optimizer, scaler, and step.
4. If an upload failed, download/copy the local checkpoint before the Kaggle session expires.

**Checkpoint uploads require Kaggle Secrets** named `KAGGLE_USERNAME` and `KAGGLE_KEY`. The target slug in the training cell is `oipranesh/cs781-phase1-mamba-checkpoints`; change it if you want a different dataset slug.
